<a href="https://colab.research.google.com/github/Amrutha-jit-acc/iiit-h-mata/blob/main/kannada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdf2image google-cloud-vision pandas PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.1/529.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 85.9 MB/s eta 0:00:00


In [ ]:
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (277 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import os

# --- Configuration ---
# 1. Update this with the EXACT name of the JSON key you uploaded
KEY_FILE_NAME = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"
# 2. Path to the folder containing your Tamil PDF textbooks
INPUT_FOLDER_DRIVE = '/content/drive/MyDrive/kannada-tb'
# 3. Path to the folder where you want to save the final .txt files
OUTPUT_FOLDER_DRIVE = '/content/drive/MyDrive/kannada-extracted-output'

# Set the environment variable for GCV authentication
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = f"/content/{KEY_FILE_NAME}"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER_DRIVE, exist_ok=True)

print(f"Authentication set using {KEY_FILE_NAME}.")
print(f"Input Folder: {INPUT_FOLDER_DRIVE}")
print(f"Output Folder: {OUTPUT_FOLDER_DRIVE}")

Authentication set using /content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json.
Input Folder: /content/drive/MyDrive/kannada-tb
Output Folder: /content/drive/MyDrive/kannada-extracted-output


In [ ]:
import os
from google.cloud import vision

def test_vision_auth():
    print("--- Starting Authentication Check ---")

    # 1. Check if the environment variable is set
    cred_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
    if cred_path:
        print(f"✅ GOOGLE_APPLICATION_CREDENTIALS found: {cred_path}")
    else:
        print("⚠️  Warning: GOOGLE_APPLICATION_CREDENTIALS environment variable is NOT set.")
        print("   (The client will attempt to use default system credentials)")

    try:
        # 2. Instantiate the client
        client = vision.ImageAnnotatorClient()

        # 3. Perform a simple API call (Label Detection on a remote image)
        # We use a public Wikipedia image to test the connection
        image_uri = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2f/Google_2015_logo.svg/368px-Google_2015_logo.svg.png"
        image = vision.Image()
        image.source.image_uri = image_uri

        print("🔄 Sending test request to Cloud Vision API...")
        response = client.label_detection(image=image)

        if response.error.message:
            raise Exception(f'{response.error.message}')

        print("\n✅ SUCCESS! Authentication is working and the API is enabled.")
        print(f"   Top label found: {response.label_annotations[0].description}")

    except Exception as e:
        print("\n❌ FAILED.")
        print(f"   Error details: {e}")
        print("\n   Troubleshooting tips:")
        print("   1. If the error says 'Project not enabled', wait a few more minutes.")
        print("   2. If the error says 'Could not automatically determine credentials', export your key file path.")

if __name__ == "__main__":
    test_vision_auth()

--- Starting Authentication Check ---
✅ GOOGLE_APPLICATION_CREDENTIALS found: /content//content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json

❌ FAILED.
   Error details: File /content//content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json was not found.

   Troubleshooting tips:
   1. If the error says 'Project not enabled', wait a few more minutes.
   2. If the error says 'Could not automatically determine credentials', export your key file path.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Kannada Textbook OCR + Exercise Extractor
PDF → Google Vision OCR → Text → Extract ಅಭ್ಯಾಸ/ಪ್ರಶ್ನೆಗಳು sections

Author: ChatGPT (customized)
"""

import os
import re
!pip install PyMuPDF # Install PyMuPDF
!pip install google-cloud-vision pillow # Ensure google-cloud-vision is installed
import fitz  # PyMuPDF
from google.cloud import vision
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
#  STEP 1 — Google OCR Key Setup
# ------------------------------------------------------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"
client = vision.ImageAnnotatorClient()

# ------------------------------------------------------------
#  STEP 2 — OCR function using Google Vision API
# ------------------------------------------------------------
def google_ocr_image(image_bytes):
    image = vision.Image(content=image_bytes)
    response = client.text_detection(image=image)

    if response.error.message:
        raise Exception(response.error.message)

    return response.full_text_annotation.text


# ------------------------------------------------------------
#  STEP 3 — Convert PDF page-by-page into text using OCR
# ------------------------------------------------------------
def pdf_to_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = []

    print(f"🔍 OCR Processing: {pdf_path}")

    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        pix = page.get_pixmap(dpi=200)

        text = google_ocr_image(pix.tobytes())
        full_text.append(text)

        print(f"   ✔ Page {page_index+1}/{len(doc)} OCR done")

    return "\n".join(full_text)


# ------------------------------------------------------------
#  STEP 4 — Ignore ಪಾಠ before ಪರಿವಿಡಿ
# ------------------------------------------------------------
def find_parividi_index(lines):
    for i, line in enumerate(lines):
        if re.search(r'ಪರಿವಿಡಿ', line):
            return i
    return 0


# ------------------------------------------------------------
#  STEP 5 — Kannada Chapter Boundary Detection
# ------------------------------------------------------------
def is_chapter_boundary(line):
    patterns = [
        r'ಪಾಠ\s*[-:.]?\s*\d+',
        r'^\s*\d+\s*[.)]?\s*ಪಾಠ',
        r'ಪಾಠ\s*[ಒಓಔಕಖಗಘಙಚ]+',
        r'^Lesson\s+\d+',
        r'^Chapter\s+\d+',
        r'={3,}',
    ]
    return any(re.search(p, line, re.IGNORECASE) for p in patterns)


# ------------------------------------------------------------
#  STEP 6 — Kannada Question Section Start Detection
# ------------------------------------------------------------
def is_question_section_start(line):
    headers = [
        r'ಅಭ್ಯಾಸ',
        r'ಪ್ರಶ್ನೆಗಳು',
        r'ಪ್ರಶ್ನೆ',
        r'ಉತ್ತರಿಸಿ',
        r'ವ್ಯಾಯಾಮ',
        r'ಚಟುವಟಿಕೆ',
        r'ಪರಿಶೀಲನೆ',
        r'Activity',
        r'Questions?',
    ]
    return any(re.search(p, line, re.IGNORECASE) for p in headers)


# ------------------------------------------------------------
#  STEP 7 — Extract questions per chapter
# ------------------------------------------------------------
def extract_questions_kannada(text_content):
    lines = text_content.split("\n")
    parividi_index = find_parividi_index(lines)

    chapters = []
    current_chapter = None
    in_question_section = False
    question_buffer = []
    consecutive_empty = 0

    i = 0
    while i < len(lines):
        line = lines[i].rstrip()
        stripped = line.strip()

        consecutive_empty = 0 if stripped else consecutive_empty + 1

        # ---------------- chapter start (AFTER ಪರಿವಿಡಿ only) ----------------
        if is_chapter_boundary(stripped) and i >= parividi_index:
            if current_chapter and question_buffer:
                chapters.append((current_chapter, "\n".join(question_buffer).strip()))

            current_chapter = stripped
            question_buffer = []
            in_question_section = False
            i += 1
            continue

        # ---------------- question section start ----------------
        if is_question_section_start(stripped):
            in_question_section = True
            question_buffer.append(line)
            consecutive_empty = 0
            i += 1
            continue

        # ---------------- collect question text ----------------
        if in_question_section:
            if consecutive_empty > 4:
                peek = i + 1
                while peek < len(lines) and not lines[peek].strip():
                    peek += 1

                if peek < len(lines) and is_chapter_boundary(lines[peek].strip()):
                    chapters.append((current_chapter, "\n".join(question_buffer).strip()))
                    question_buffer = []
                    in_question_section = False
                    i += 1
                    continue

            if consecutive_empty <= 3:
                question_buffer.append(line)

        i += 1

    # save last chapter
    if current_chapter and question_buffer:
        chapters.append((current_chapter, "\n".join(question_buffer).strip()))

    return chapters


# ------------------------------------------------------------
#  STEP 8 — Clean text formatting
# ------------------------------------------------------------
def clean_questions(text):
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    text = re.sub(r'(?:^|\n)\s*(Page|ಪುಟ)\s*\d+\s*(?:\n|$)', '\n', text, flags=re.IGNORECASE)
    return "\n".join(line.rstrip() for line in text.split("\n")).strip()


# ------------------------------------------------------------
#  STEP 9 — Full pipeline (PDF folder → output folder)
# ------------------------------------------------------------
def process_kannada_textbooks(pdf_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    if not pdf_files:
        print("❌ No PDF files found.")
        return

    for pdf_path in pdf_files:
        print(f"\n📘 Processing textbook: {pdf_path.name}")

        text_content = pdf_to_text(pdf_path)

        chapters = extract_questions_kannada(text_content)
        out_file = os.path.join(output_folder, f"{pdf_path.stem}_questions.txt")

        with open(out_file, "w", encoding="utf-8") as f:
            f.write(f"ಮೂಲ PDF: {pdf_path.name}\n")
            f.write(f"ದಿನಾಂಕ: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"ಒಟ್ಟ ಅಧ್ಯಾಯಗಳು: {len(chapters)}\n")
            f.write("="*70 + "\n\n")

            for i, (chapter, data) in enumerate(chapters, 1):
                f.write(f"\n{'='*70}\n")
                f.write(f"[{i}/{len(chapters)}] {chapter}\n")
                f.write(f"{'='*70}\n\n")
                f.write(clean_questions(data))
                f.write("\n\n")

        print(f"✔ Saved: {out_file}")


# ------------------------------------------------------------
#  MAIN
# ------------------------------------------------------------
def main():
    pdf_folder = "/content/drive/MyDrive/kannada-tb"
    output_folder = "/content/drive/MyDrive/Kannada_Questions_Output"

    process_kannada_textbooks(pdf_folder, output_folder)
    print("\n✨ All done!")


if __name__ == "__main__":
    main()



📘 Processing textbook: 1std Kan FL Part- 1 2025-26.pdf
🔍 OCR Processing: /content/drive/MyDrive/kannada-tb/1std Kan FL Part- 1 2025-26.pdf
   ✔ Page 1/104 OCR done
   ✔ Page 2/104 OCR done
   ✔ Page 3/104 OCR done
   ✔ Page 4/104 OCR done
   ✔ Page 5/104 OCR done
   ✔ Page 6/104 OCR done
   ✔ Page 7/104 OCR done
   ✔ Page 8/104 OCR done
   ✔ Page 9/104 OCR done
   ✔ Page 10/104 OCR done
   ✔ Page 11/104 OCR done
   ✔ Page 12/104 OCR done
   ✔ Page 13/104 OCR done
   ✔ Page 14/104 OCR done
   ✔ Page 15/104 OCR done
   ✔ Page 16/104 OCR done
   ✔ Page 17/104 OCR done
   ✔ Page 18/104 OCR done
   ✔ Page 19/104 OCR done
   ✔ Page 20/104 OCR done
   ✔ Page 21/104 OCR done
   ✔ Page 22/104 OCR done
   ✔ Page 23/104 OCR done
   ✔ Page 24/104 OCR done
   ✔ Page 25/104 OCR done
   ✔ Page 26/104 OCR done
   ✔ Page 27/104 OCR done
   ✔ Page 28/104 OCR done
   ✔ Page 29/104 OCR done
   ✔ Page 30/104 OCR done
   ✔ Page 31/104 OCR done
   ✔ Page 32/104 OCR done
   ✔ Page 33/104 OCR done
   ✔ Page 3

the above code performed okay with

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Kannada Textbook OCR + Exercise Extractor
PDF → Google Vision OCR → Text → Extract ಅಭ್ಯಾಸ/ಪ್ರಶ್ನೆಗಳು sections

Author: ChatGPT (customized)
"""

import os
import re
!pip install PyMuPDF # Install PyMuPDF
!pip install google-cloud-vision pillow # Ensure google-cloud-vision is installed
import fitz  # PyMuPDF
from google.cloud import vision
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
#  STEP 1 — Google OCR Key Setup
# ------------------------------------------------------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"
client = vision.ImageAnnotatorClient()

# ------------------------------------------------------------
#  STEP 2 — OCR function using Google Vision API
# ------------------------------------------------------------
def google_ocr_image(image_bytes):
    image = vision.Image(content=image_bytes)
    response = client.text_detection(image=image)

    if response.error.message:
        raise Exception(response.error.message)

    return response.full_text_annotation.text


# ------------------------------------------------------------
#  STEP 3 — Convert PDF page-by-page into text using OCR
# ------------------------------------------------------------
def pdf_to_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = []

    print(f"🔍 OCR Processing: {pdf_path}")

    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        pix = page.get_pixmap(dpi=200)

        text = google_ocr_image(pix.tobytes())
        full_text.append(text)

        print(f"   ✔ Page {page_index+1}/{len(doc)} OCR done")

    return "\n".join(full_text)


# ------------------------------------------------------------
#  STEP 4 — Ignore ಪಾಠ before ಪರಿವಿಡಿ
# ------------------------------------------------------------
def find_parividi_index(lines):
    for i, line in enumerate(lines):
        if 'ಪರಿವಿಡಿ' in line:
            print(f"✔ Found ಪರಿವಿಡಿ at line {i}")
            return i
    print("⚠ ಪರಿವಿಡಿ not found, starting from beginning")
    return 0


# ------------------------------------------------------------
#  STEP 5 — Kannada Chapter Boundary Detection (more flexible)
# ------------------------------------------------------------
def is_chapter_boundary(line):
    line_clean = line.strip()

    # Check for ಪಾಠ with number
    if 'ಪಾಠ' in line_clean:
        # Check if there's a number nearby
        if re.search(r'\d+', line_clean):
            return True
        # Check for Kannada numerals
        if re.search(r'[೦೧೨೩೪೫೬೭೮೯]+', line_clean):
            return True

    # Check for Lesson/Chapter
    if re.search(r'\b(Lesson|Chapter)\s+\d+', line_clean, re.IGNORECASE):
        return True

    # Check for parenthesis pattern like (ಫಾಠ - 2 : ...)
    if re.search(r'\(.*ಫಾಠ.*[-–—].*\d+', line_clean):
        return True

    return False


# ------------------------------------------------------------
#  STEP 6 — Kannada Question Section Start Detection (more flexible)
# ------------------------------------------------------------
def is_question_section_start(line):
    line_clean = line.strip()

    keywords = ['ಅಭ್ಯಾಸ', 'ಅಭ್ಯಾಾಸ', 'ಪ್ರಶ್ನೆಗಳು', 'ಪ್ರಶ್ನೆ', 'ಉತ್ತರಿಸಿ',
                'ವ್ಯಾಯಾಮ', 'ಚಟುವಟಿಕೆ', 'ಪರಿಶೀಲನೆ', 'Activity', 'Questions', 'Question']

    for keyword in keywords:
        if keyword in line_clean:
            return True

    return False


# ------------------------------------------------------------
#  STEP 7 — Extract questions per chapter
# ------------------------------------------------------------
def extract_questions_kannada(text_content):
    lines = text_content.split("\n")
    parividi_index = find_parividi_index(lines)

    chapters = []
    current_chapter = None
    in_question_section = False
    question_buffer = []
    chapter_count = 0

    for i in range(parividi_index, len(lines)):
        line = lines[i]
        stripped = line.strip()

        # Skip empty lines
        if not stripped:
            if in_question_section:
                question_buffer.append(line)
            continue

        # ---------------- chapter start ----------------
        if is_chapter_boundary(stripped):
            # Save previous chapter if exists
            if current_chapter is not None and question_buffer:
                chapters.append((current_chapter, "\n".join(question_buffer).strip()))
                print(f"   💾 Saved questions for: {current_chapter[:50]}...")

            chapter_count += 1
            current_chapter = stripped
            print(f"\n📖 Chapter {chapter_count} found: {stripped[:60]}...")
            question_buffer = []
            in_question_section = False
            continue

        # ---------------- question section start ----------------
        if is_question_section_start(stripped):
            if current_chapter is not None:
                in_question_section = True
                print(f"   ❓ Question section started: {stripped[:40]}...")
                question_buffer.append(line)
            continue

        # ---------------- collect all text after question section starts ----------------
        if in_question_section:
            question_buffer.append(line)

    # save last chapter
    if current_chapter is not None and question_buffer:
        chapters.append((current_chapter, "\n".join(question_buffer).strip()))
        print(f"   💾 Saved questions for: {current_chapter[:50]}...")

    print(f"\n✅ Total chapters extracted: {len(chapters)}")
    return chapters


# ------------------------------------------------------------
#  STEP 8 — Clean text formatting
# ------------------------------------------------------------
def clean_questions(text):
    # Remove excessive empty lines
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)

    # Remove page numbers
    text = re.sub(r'(?:^|\n)\s*(Page|ಪುಟ)\s*\d+\s*(?:\n|$)', '\n', text, flags=re.IGNORECASE)

    # Remove standalone numbers (likely page numbers)
    text = re.sub(r'(?:^|\n)\s*\d{1,3}\s*(?:\n|$)', '\n', text)

    return "\n".join(line.rstrip() for line in text.split("\n")).strip()


# ------------------------------------------------------------
#  STEP 9 — Full pipeline (PDF folder → output folder)
# ------------------------------------------------------------
def process_kannada_textbooks(pdf_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    if not pdf_files:
        print("❌ No PDF files found.")
        return

    for pdf_path in pdf_files:
        print(f"\n{'='*70}")
        print(f"📘 Processing textbook: {pdf_path.name}")
        print(f"{'='*70}")

        text_content = pdf_to_text(pdf_path)

        chapters = extract_questions_kannada(text_content)
        out_file = os.path.join(output_folder, f"{pdf_path.stem}_questions.txt")

        with open(out_file, "w", encoding="utf-8") as f:
            f.write(f"ಮೂಲ PDF: {pdf_path.name}\n")
            f.write(f"ದಿನಾಂಕ: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"ಒಟ್ಟ ಅಧ್ಯಾಯಗಳು: {len(chapters)}\n")
            f.write("="*70 + "\n\n")

            for i, (chapter, data) in enumerate(chapters, 1):
                f.write(f"\n{'='*70}\n")
                f.write(f"[{i}/{len(chapters)}] {chapter}\n")
                f.write(f"{'='*70}\n\n")
                f.write(clean_questions(data))
                f.write("\n\n")

        print(f"\n✔ Saved: {out_file}")
        print(f"✔ Extracted {len(chapters)} chapters with questions\n")


# ------------------------------------------------------------
#  MAIN
# ------------------------------------------------------------
def main():
    pdf_folder = "/content/drive/MyDrive/kannada-tb"
    output_folder = "/content/drive/MyDrive/Kannada_Ques"

    process_kannada_textbooks(pdf_folder, output_folder)
    print("\n✨ All done!")


if __name__ == "__main__":
    main()


📘 Processing textbook: 1std Kan FL Part- 1 2025-26.pdf
🔍 OCR Processing: /content/drive/MyDrive/kannada-tb/1std Kan FL Part- 1 2025-26.pdf
   ✔ Page 1/104 OCR done
   ✔ Page 2/104 OCR done
   ✔ Page 3/104 OCR done
   ✔ Page 4/104 OCR done
   ✔ Page 5/104 OCR done
   ✔ Page 6/104 OCR done
   ✔ Page 7/104 OCR done
   ✔ Page 8/104 OCR done
   ✔ Page 9/104 OCR done
   ✔ Page 10/104 OCR done
   ✔ Page 11/104 OCR done
   ✔ Page 12/104 OCR done
   ✔ Page 13/104 OCR done
   ✔ Page 14/104 OCR done
   ✔ Page 15/104 OCR done
   ✔ Page 16/104 OCR done
   ✔ Page 17/104 OCR done
   ✔ Page 18/104 OCR done
   ✔ Page 19/104 OCR done
   ✔ Page 20/104 OCR done
   ✔ Page 21/104 OCR done
   ✔ Page 22/104 OCR done
   ✔ Page 23/104 OCR done
   ✔ Page 24/104 OCR done
   ✔ Page 25/104 OCR done
   ✔ Page 26/104 OCR done
   ✔ Page 27/104 OCR done
   ✔ Page 28/104 OCR done
   ✔ Page 29/104 OCR done
   ✔ Page 30/104 OCR done
   ✔ Page 31/104 OCR done
   ✔ Page 32/104 OCR done
   ✔ Page 33/104 OCR done
   ✔ Page 3

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Alternative: Manual-assisted Kannada Textbook Exercise Extractor
Saves full OCR text with line numbers for manual review, then extracts based on line ranges
"""

import os
import re
!pip install PyMuPDF
!pip install google-cloud-vision pillow
import fitz
from google.cloud import vision
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
#  Google OCR Setup
# ------------------------------------------------------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"
client = vision.ImageAnnotatorClient()

def google_ocr_image(image_bytes):
    image = vision.Image(content=image_bytes)
    response = client.text_detection(image=image)
    if response.error.message:
        raise Exception(response.error.message)
    return response.full_text_annotation.text

# ------------------------------------------------------------
#  ALTERNATIVE METHOD 1: Save full OCR with line numbers
# ------------------------------------------------------------
def save_full_ocr_with_line_numbers(pdf_path, output_folder):
    """Save complete OCR text with line numbers for manual review"""
    doc = fitz.open(pdf_path)
    full_text = []

    print(f"🔍 OCR Processing: {pdf_path.name}")

    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        pix = page.get_pixmap(dpi=200)
        text = google_ocr_image(pix.tobytes())

        full_text.append(f"\n{'='*70}\n")
        full_text.append(f"PAGE {page_index+1}\n")
        full_text.append(f"{'='*70}\n")
        full_text.append(text)

        print(f"   ✔ Page {page_index+1}/{len(doc)} done")

    # Save with line numbers
    out_file = os.path.join(output_folder, f"{pdf_path.stem}_FULL_OCR.txt")
    with open(out_file, "w", encoding="utf-8") as f:
        content = "\n".join(full_text)
        lines = content.split("\n")
        for i, line in enumerate(lines, 1):
            f.write(f"{i:5d} | {line}\n")

    print(f"✔ Saved full OCR: {out_file}")
    return out_file

# ------------------------------------------------------------
#  ALTERNATIVE METHOD 2: Extract based on keyword frequency analysis
# ------------------------------------------------------------
def analyze_keyword_positions(pdf_path, output_folder):
    """Analyze where keywords appear and their context"""
    doc = fitz.open(pdf_path)

    keywords = ['ಅಭ್ಯಾಸ', 'ಅಭ್ಯಾಾಸ', 'ಪ್ರಶ್ನೆಗಳು', 'ಪಾಠ', 'ಫಾಠ', 'Lesson', 'Chapter']
    keyword_positions = {kw: [] for kw in keywords}

    print(f"🔍 Analyzing: {pdf_path.name}")

    all_text = []
    for page_index in range(len(doc)):
        page = doc.load_page(page_index)
        pix = page.get_pixmap(dpi=200)
        text = google_ocr_image(pix.tobytes())
        all_text.append((page_index + 1, text))
        print(f"   ✔ Page {page_index+1}/{len(doc)} analyzed")

    # Find all keyword occurrences
    for page_num, text in all_text:
        lines = text.split("\n")
        for line_num, line in enumerate(lines, 1):
            for keyword in keywords:
                if keyword in line:
                    keyword_positions[keyword].append({
                        'page': page_num,
                        'line': line_num,
                        'text': line.strip()[:100]
                    })

    # Save analysis
    out_file = os.path.join(output_folder, f"{pdf_path.stem}_KEYWORD_ANALYSIS.txt")
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(f"Keyword Analysis for: {pdf_path.name}\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*70 + "\n\n")

        for keyword, positions in keyword_positions.items():
            if positions:
                f.write(f"\n{'='*70}\n")
                f.write(f"Keyword: {keyword} (Found {len(positions)} times)\n")
                f.write(f"{'='*70}\n\n")
                for pos in positions:
                    f.write(f"  Page {pos['page']}, Line {pos['line']}: {pos['text']}\n")

    print(f"✔ Saved analysis: {out_file}")
    return keyword_positions

# ------------------------------------------------------------
#  ALTERNATIVE METHOD 3: Extract by page ranges
# ------------------------------------------------------------
def extract_by_page_ranges(pdf_path, output_folder, page_ranges):
    """
    Extract specific page ranges
    page_ranges = [(start_page, end_page, "Chapter Name"), ...]
    Example: [(15, 18, "ಪಾಠ 1"), (25, 28, "ಪಾಠ 2")]
    """
    doc = fitz.open(pdf_path)

    print(f"🔍 Extracting specified ranges from: {pdf_path.name}")

    extractions = []
    for start_page, end_page, chapter_name in page_ranges:
        chapter_text = []
        for page_index in range(start_page - 1, end_page):
            if page_index < len(doc):
                page = doc.load_page(page_index)
                pix = page.get_pixmap(dpi=200)
                text = google_ocr_image(pix.tobytes())
                chapter_text.append(text)
                print(f"   ✔ Extracted page {page_index+1}")

        extractions.append((chapter_name, "\n".join(chapter_text)))

    # Save extractions
    out_file = os.path.join(output_folder, f"{pdf_path.stem}_EXTRACTED_RANGES.txt")
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(f"ಮೂಲ PDF: {pdf_path.name}\n")
        f.write(f"ದಿನಾಂಕ: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*70 + "\n\n")

        for chapter_name, text in extractions:
            f.write(f"\n{'='*70}\n")
            f.write(f"{chapter_name}\n")
            f.write(f"{'='*70}\n\n")
            f.write(text)
            f.write("\n\n")

    print(f"✔ Saved: {out_file}")

# ------------------------------------------------------------
#  MAIN - Choose your method
# ------------------------------------------------------------
def main():
    pdf_folder = "/content/drive/MyDrive/kannada-tb"
    output_folder = "/content/drive/MyDrive/Questions_Output"
    os.makedirs(output_folder, exist_ok=True)

    pdf_files = sorted(list(Path(pdf_folder).glob("*.pdf")))

    print("\n" + "="*70)
    print("ALTERNATIVE EXTRACTION METHODS")
    print("="*70)
    print("\nChoose method:")
    print("1. Save FULL OCR with line numbers (for manual review)")
    print("2. Analyze keyword positions (find where ಅಭ್ಯಾಸ appears)")
    print("3. Extract specific page ranges (manual input)")
    print("="*70 + "\n")

    # METHOD 1: Save everything with line numbers
    print("\n🔹 Running METHOD 1: Saving full OCR...\n")
    for pdf_path in pdf_files:
        save_full_ocr_with_line_numbers(pdf_path, output_folder)

    # METHOD 2: Analyze where keywords appear
    print("\n🔹 Running METHOD 2: Analyzing keywords...\n")
    for pdf_path in pdf_files:
        analyze_keyword_positions(pdf_path, output_folder)

    print("\n✨ Analysis complete!")
    print("\nNEXT STEPS:")
    print("1. Open the *_FULL_OCR.txt files to see all text with line numbers")
    print("2. Open the *_KEYWORD_ANALYSIS.txt files to see where ಅಭ್ಯಾಸ appears")
    print("3. Use this info to identify patterns or page ranges")
    print("4. Then use METHOD 3 with specific page ranges if needed")

    # EXAMPLE for METHOD 3 (uncomment and customize):
    # page_ranges = [
    #     (15, 18, "ಪಾಠ 1 - ಅಭ್ಯಾಸ"),
    #     (25, 28, "ಪಾಠ 2 - ಅಭ್ಯಾಸ"),
    # ]
    # extract_by_page_ranges(pdf_files[0], output_folder, page_ranges)

if __name__ == "__main__":
    main()


ALTERNATIVE EXTRACTION METHODS

Choose method:
1. Save FULL OCR with line numbers (for manual review)
2. Analyze keyword positions (find where ಅಭ್ಯಾಸ appears)
3. Extract specific page ranges (manual input)


🔹 Running METHOD 1: Saving full OCR...

🔍 OCR Processing: 10th Kan FL  Part-1 2025-26.pdf
   ✔ Page 1/104 done
   ✔ Page 2/104 done
   ✔ Page 3/104 done
   ✔ Page 4/104 done
   ✔ Page 5/104 done
   ✔ Page 6/104 done
   ✔ Page 7/104 done
   ✔ Page 8/104 done
   ✔ Page 9/104 done
   ✔ Page 10/104 done
   ✔ Page 11/104 done
   ✔ Page 12/104 done
   ✔ Page 13/104 done
   ✔ Page 14/104 done
   ✔ Page 15/104 done
   ✔ Page 16/104 done
   ✔ Page 17/104 done
   ✔ Page 18/104 done
   ✔ Page 19/104 done
   ✔ Page 20/104 done
   ✔ Page 21/104 done
   ✔ Page 22/104 done
   ✔ Page 23/104 done
   ✔ Page 24/104 done
   ✔ Page 25/104 done
   ✔ Page 26/104 done
   ✔ Page 27/104 done
   ✔ Page 28/104 done
   ✔ Page 29/104 done
   ✔ Page 30/104 done
   ✔ Page 31/104 done
   ✔ Page 32/104 done
   ✔ P

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
METHOD 3: Extract Kannada exercises by specific page ranges
Based on keyword analysis showing ಅಭ್ಯಾಸ appears on pages: 19, 24, 28, 33, 47, 78, 82, 88, 92, 95
"""

import os
import re
import fitz
from google.cloud import vision
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
#  Google OCR Setup
# ------------------------------------------------------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"
client = vision.ImageAnnotatorClient()

def google_ocr_image(image_bytes):
    image = vision.Image(content=image_bytes)
    response = client.text_detection(image=image)
    if response.error.message:
        raise Exception(response.error.message)
    return response.full_text_annotation.text

# ------------------------------------------------------------
#  Extract specific page ranges
# ------------------------------------------------------------
def extract_by_page_ranges(pdf_path, output_folder, page_ranges):
    """
    Extract specific page ranges
    page_ranges = [(start_page, end_page, "Chapter Name"), ...]
    """
    doc = fitz.open(pdf_path)

    print(f"🔍 Extracting specified ranges from: {pdf_path.name}")

    extractions = []
    for start_page, end_page, chapter_name in page_ranges:
        chapter_text = []
        print(f"\n📖 Extracting: {chapter_name} (Pages {start_page}-{end_page})")

        for page_index in range(start_page - 1, min(end_page, len(doc))):
            page = doc.load_page(page_index)
            pix = page.get_pixmap(dpi=200)
            text = google_ocr_image(pix.tobytes())
            chapter_text.append(text)
            print(f"   ✔ Extracted page {page_index+1}")

        extractions.append((chapter_name, "\n".join(chapter_text)))

    # Save extractions
    out_file = os.path.join(output_folder, f"{pdf_path.stem}_EXERCISES.txt")
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(f"ಮೂಲ PDF: {pdf_path.name}\n")
        f.write(f"ದಿನಾಂಕ: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"ಒಟ್ಟ ಅಭ್ಯಾಸಗಳು: {len(extractions)}\n")
        f.write("="*70 + "\n\n")

        for chapter_name, text in extractions:
            f.write(f"\n{'='*70}\n")
            f.write(f"{chapter_name}\n")
            f.write(f"{'='*70}\n\n")
            f.write(clean_text(text))
            f.write("\n\n")

    print(f"\n✔ Saved: {out_file}")
    return out_file

def clean_text(text):
    # Remove excessive empty lines
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    # Remove page numbers
    text = re.sub(r'(?:^|\n)\s*(Page|ಪುಟ)\s*\d+\s*(?:\n|$)', '\n', text, flags=re.IGNORECASE)
    return text.strip()

# ------------------------------------------------------------
#  MAIN
# ------------------------------------------------------------
def main():
    pdf_folder = "/content/drive/MyDrive/kannada-tb"
    output_folder = "/content/drive/MyDrive/Kannada_Question1"
    os.makedirs(output_folder, exist_ok=True)

    # Based on your keyword analysis, ಅಭ್ಯಾಸ appears on these pages:
    # Pages: 19, 24, 28, 33, 47, 78, 82, 88, 92, 95
    # I'm extracting a few pages after each ಅಭ್ಯಾಸ to capture the full exercise

    page_ranges_2std = [
        (17, 19, "ಪಾಠ 0 - ಅಭ್ಯಾಸ"),
        (22, 23, "ಪಾಠ 1 - ಅಭ್ಯಾಸ"),
        (26, 27, "ಪಾಠ 2 - ಅಭ್ಯಾಸ"),
        (31, 32, "ಪಾಠ 2 - ಅಭ್ಯಾಸ"),
        (38, 41, "ಪಾಠ 2 - ಅಭ್ಯಾಸ"),
        (44, 45, "ಪಾಠ 5- ಅಭ್ಯಾಸ"),
        (51, 52, "ಪಾಠ 6- ಅಭ್ಯಾಸ"),
        (57, 58, "ಪಾಠ 7- ಅಭ್ಯಾಸ"),
        (63, 64, "ಪಾಠ 8- ಅಭ್ಯಾಸ"),

    ]

    # Process the 1st standard PDF
    pdf_path = Path(pdf_folder) / "2nd  Kan FL 2025 Part-1 2025-26.pdf"

    if pdf_path.exists():
        extract_by_page_ranges(pdf_path, output_folder, page_ranges_2std)
    else:
        print(f"❌ PDF not found: {pdf_path}")

    print("\n✨ Extraction complete!")
    print("\n📝 NEXT STEPS:")
    print("1. Review the extracted exercises")
    print("2. Adjust page ranges if needed (some exercises may span more pages)")
    print("3. Apply the same method to other PDFs after analyzing their keyword positions")

if __name__ == "__main__":
    main()

🔍 Extracting specified ranges from: 2nd  Kan FL 2025 Part-1 2025-26.pdf

📖 Extracting: ಪಾಠ 0 - ಅಭ್ಯಾಸ (Pages 17-19)
   ✔ Extracted page 17
   ✔ Extracted page 18
   ✔ Extracted page 19

📖 Extracting: ಪಾಠ 1 - ಅಭ್ಯಾಸ (Pages 22-23)
   ✔ Extracted page 22
   ✔ Extracted page 23

📖 Extracting: ಪಾಠ 2 - ಅಭ್ಯಾಸ (Pages 26-27)
   ✔ Extracted page 26
   ✔ Extracted page 27

📖 Extracting: ಪಾಠ 2 - ಅಭ್ಯಾಸ (Pages 31-32)
   ✔ Extracted page 31
   ✔ Extracted page 32

📖 Extracting: ಪಾಠ 2 - ಅಭ್ಯಾಸ (Pages 38-41)
   ✔ Extracted page 38
   ✔ Extracted page 39
   ✔ Extracted page 40
   ✔ Extracted page 41

📖 Extracting: ಪಾಠ 5- ಅಭ್ಯಾಸ (Pages 44-45)
   ✔ Extracted page 44
   ✔ Extracted page 45

📖 Extracting: ಪಾಠ 6- ಅಭ್ಯಾಸ (Pages 51-52)
   ✔ Extracted page 51
   ✔ Extracted page 52

📖 Extracting: ಪಾಠ 7- ಅಭ್ಯಾಸ (Pages 57-58)
   ✔ Extracted page 57
   ✔ Extracted page 58

📖 Extracting: ಪಾಠ 8- ಅಭ್ಯಾಸ (Pages 63-64)
   ✔ Extracted page 63
   ✔ Extracted page 64

✔ Saved: /content/drive/MyDrive/Kannada_Question1/2n

In [ ]:
import os
import io
import re
from google.cloud import vision
from pdf2image import convert_from_path

# =================CONFIGURATION=================
# Replace with the path to your Google JSON key file
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"

# List of keywords that signify the START of an exercise section
START_MARKERS = [
    r'ಅಭ್ಯಾಸ',
    r'ಅಭ್ಯಾಸಗಳು'
]

# List of keywords that signify the END of an exercise section
# (Next chapter, or non-question sections like Projects/Games)
END_MARKERS = [
    r'ಪಾಠ',
    r'ಪದ್ಯ',
    r'ಗದ್ಯ',
    r'ಪೂರಕ',
    r'ಯೋಜನೆ',
    r'ಭಾಷಾ ಆಟ',
    r'ವ್ಯಾಕರಣ'
]
# ===============================================

def get_text_from_pdf(pdf_path):
    """
    Converts PDF to images and uses Google Vision API to extract text.
    """
    client = vision.ImageAnnotatorClient()

    print(f"Processing PDF: {pdf_path}...")
    # Convert PDF to images (300 DPI is good for OCR)
    try:
        images = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        print(f"Error converting PDF. Is Poppler installed? {e}")
        return ""

    full_text = ""

    for i, image in enumerate(images):
        print(f"OCR-ing page {i + 1}...")

        # Convert PIL image to bytes for Google API
        img_byte_arr = io.BytesIO()
        image.save(img_byte_arr, format='JPEG')
        content = img_byte_arr.getvalue()

        image_obj = vision.Image(content=content)

        # Perform Text Detection
        response = client.document_text_detection(image=image_obj)
        if response.full_text_annotation:
            full_text += response.full_text_annotation.text + "\n"

    return full_text

def extract_questions(full_text):
    """
    Parses the full text to find blocks between 'Abhyasa' and the next section,
    then extracts specific question lines.
    """
    lines = full_text.split('\n')
    extracted_questions = []
    is_in_exercise_section = False

    # Regex to identify lines that look like questions/bullets
    # Matches: 1., 1), ೧., ೧), ಅ., ಅ), etc.
    question_pattern = re.compile(r'^\s*([0-9೦-೯]+|[ಅ-ಹ]+)[\.\)]\s+(.*)')

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Check if we are entering an exercise section
        if any(re.search(marker, line) for marker in START_MARKERS):
            is_in_exercise_section = True
            continue # Skip the header line itself

        # Check if we are leaving an exercise section (hitting a new chapter/section)
        # We only check this if we are currently INSIDE an exercise section
        if is_in_exercise_section:
            # If the line is just a very short header or matches end markers specifically
            if any(re.match(f"^{marker}", line) for marker in END_MARKERS):
                is_in_exercise_section = False
                continue

        # If we are inside the section, look for question patterns
        if is_in_exercise_section:
            match = question_pattern.match(line)
            if match:
                # match.group(0) is the whole line
                # You can filter out very short lines if they are OCR noise
                if len(line) > 5:
                    extracted_questions.append(line)

            # Optional: Capture sub-headings inside exercises (like "Answer in one word")
            # else:
            #    # Logic to capture instruction lines if needed
            #    pass

    return extracted_questions

def main():
    # Example usage
    pdf_file = "/content/drive/MyDrive/kannada-tb/10th Kan FL  Part-1 2025-26.pdf" # Replace with your PDF path

    if not os.path.exists(pdf_file):
        print(f"File {pdf_file} not found.")
        return

    # 1. Get raw text using Google OCR
    raw_text = get_text_from_pdf(pdf_file)

    # 2. Extract specific questions
    questions = extract_questions(raw_text)

    # 3. Save or Print results
    print("\n--- Extracted Questions ---\n")
    with open("extracted_questions.txt", "w", encoding="utf-8") as f:
        for q in questions:
            print(q)
            f.write(q + "\n")

    print(f"\nSaved {len(questions)} questions to extracted_questions.txt")

if __name__ == "__main__":
    main()

Processing PDF: /content/drive/MyDrive/kannada-tb/10th Kan FL  Part-1 2025-26.pdf...
OCR-ing page 1...
OCR-ing page 2...
OCR-ing page 3...
OCR-ing page 4...
OCR-ing page 5...
OCR-ing page 6...
OCR-ing page 7...
OCR-ing page 8...
OCR-ing page 9...
OCR-ing page 10...
OCR-ing page 11...
OCR-ing page 12...
OCR-ing page 13...
OCR-ing page 14...
OCR-ing page 15...
OCR-ing page 16...
OCR-ing page 17...
OCR-ing page 18...
OCR-ing page 19...
OCR-ing page 20...
OCR-ing page 21...
OCR-ing page 22...
OCR-ing page 23...
OCR-ing page 24...
OCR-ing page 25...
OCR-ing page 26...
OCR-ing page 27...
OCR-ing page 28...
OCR-ing page 29...
OCR-ing page 30...
OCR-ing page 31...
OCR-ing page 32...
OCR-ing page 33...
OCR-ing page 34...
OCR-ing page 35...
OCR-ing page 36...
OCR-ing page 37...
OCR-ing page 38...
OCR-ing page 39...
OCR-ing page 40...
OCR-ing page 41...
OCR-ing page 42...
OCR-ing page 43...
OCR-ing page 44...
OCR-ing page 45...
OCR-ing page 46...
OCR-ing page 47...
OCR-ing page 48...
OCR-ing page

THE BELOW CODES ARE FINAL FOR THE KANNADA QUESTION EXTRACTION

In [ ]:
import os
import io
import re
import csv
import glob
from google.cloud import vision
from pdf2image import convert_from_path

# ================= CONFIGURATION =================
# 1. Path to your Google Cloud JSON key
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"

# 2. Folder containing your input PDFs
INPUT_FOLDER = "/content/drive/MyDrive/kannada-tb"

# 3. Output filename
OUTPUT_CSV = "/content/drive/MyDrive/newkannada_questions_dataset.csv"

# ================= LOGIC CONSTANTS =================
# Keywords that start an exercise section
START_MARKERS = [
    r'ಅಭ್ಯಾಸ', r'ಅಭ್ಯಾಸಗಳು'
]

# Keywords that definitely end a chapter's exercise section
# We removed 'ವ್ಯಾಕರಣ' so grammar is included.
END_MARKERS = [
    r'^ಪಾಠ\s+[0-9೦-೯]+',  # Matches "Patha 1"
    r'^ಪದ್ಯ\s+[0-9೦-೯]+',  # Matches "Padya 1"
    r'^ಗದ್ಯ\s+[0-9೦-೯]+',  # Matches "Gadya 1"
    r'^ಘಟಕ\s+[0-9೦-೯]+',  # Matches "Ghataka 1" (Unit)
    r'^ಪೂರಕ\s+ಓದು',       # Supplementary Reading
    r'^ಯೋಜನೆ',            # Projects (usually skipping projects, keep if needed)
]

# Regex to find Main Section Headers (e.g., "Answer the following", "Fill in the blanks")
# These usually don't start with numbers, or start with main letters like ಅ, ಆ, ಇ
SECTION_HEADER_PATTERN = re.compile(r'^\s*([ಅ-ಋa-zA-Z]+)[\.\)]\s+(.*)')

# Regex to find specific Questions (numbered 1, 2, 3 or ೧, ೨, ೩)
QUESTION_PATTERN = re.compile(r'^\s*([\d0-9೦-೯]+)[\.\)]\s+(.*)')

# Regex to find matching options inside a line (e.g., (a) option (b) option)
OPTION_PATTERN = re.compile(r'[\(\[]([ಅ-ಋa-dA-D]+)[\)\]]\s+([^(\[]+)')

# ===============================================

def get_text_from_pdf(pdf_path):
    """ Converts PDF pages to images and extracts text using Google Vision. """
    client = vision.ImageAnnotatorClient()
    text_data = [] # List of tuples (page_num, text)

    print(f" Converting {os.path.basename(pdf_path)} to images...")
    try:
        # DPI 300 is optimal for OCR
        images = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        print(f"Error converting PDF: {e}")
        return []

    print(f" Extracting text from {len(images)} pages...")
    for i, image in enumerate(images):
        img_byte_arr = io.BytesIO()
        image.save(img_byte_arr, format='JPEG')
        content = img_byte_arr.getvalue()
        image_obj = vision.Image(content=content)

        response = client.document_text_detection(image=image_obj)
        if response.full_text_annotation:
            text_data.append((i + 1, response.full_text_annotation.text))

    return text_data

def clean_ocr_text(text):
    """ Basic cleanup of OCR artifacts. """
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        # Remove very short junk lines (less than 2 chars)
        if len(line) > 2:
            cleaned_lines.append(line)
    return cleaned_lines

def parse_questions(cleaned_lines, filename):
    extracted_data = []

    is_in_exercise = False
    current_context = "General" # Stores headers like "Answer in one word"
    current_page = 0

    for line in cleaned_lines:
        # 1. Check for Exercise Start
        if any(re.search(m, line) for m in START_MARKERS):
            is_in_exercise = True
            current_context = "General Exercise"
            continue # Skip the word "Abhyasa" itself

        # 2. Check for Chapter End
        if any(re.search(m, line) for m in END_MARKERS):
            is_in_exercise = False
            current_context = "General"
            continue

        if not is_in_exercise:
            continue

        # 3. Detect Section Headers (e.g., "ಅ. ಈ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸಿ")
        # These help give context to single-word questions.
        header_match = SECTION_HEADER_PATTERN.match(line)
        question_match = QUESTION_PATTERN.match(line)

        # Logic: If it looks like a header (starts with letters usually)
        if header_match and not question_match:
            # Update context, remove the numbering bullet (ಅ.) to keep it clean
            current_context = header_match.group(2).strip()
            continue

        # 4. Detect Specific Questions (starts with numbers 1, 2, ೧, ೨)
        if question_match:
            q_num = question_match.group(1)
            q_text = question_match.group(2)

            # --- MCQ PARSING ---
            options = []
            # Check if this line contains options like (a) ... (b) ...
            opt_matches = OPTION_PATTERN.findall(q_text)

            clean_q_text = q_text
            if opt_matches:
                # Remove options from the main question text
                clean_q_text = re.split(r'[\(\[]([ಅ-ಋa-dA-D]+)[\)\]]', q_text)[0].strip()
                options = [f"({m[0]}) {m[1].strip()}" for m in opt_matches]

            # --- SINGLE WORD FIX ---
            # If the question is very short (e.g., "1. Sky"), prepend the context
            # Example: "Write Meanings: Sky"
            final_question = clean_q_text
            if len(clean_q_text.split()) < 3 and current_context != "General":
                final_question = f"{current_context}: {clean_q_text}"

            # Structure the row
            row = {
                'Filename': filename,
                'Section_Context': current_context,
                'Q_No': q_num,
                'Question': final_question,
                'Option_1': options[0] if len(options) > 0 else "",
                'Option_2': options[1] if len(options) > 1 else "",
                'Option_3': options[2] if len(options) > 2 else "",
                'Option_4': options[3] if len(options) > 3 else "",
            }
            extracted_data.append(row)

    return extracted_data

def main():
    # Setup CSV file
    csv_headers = ['Filename', 'Section_Context', 'Q_No', 'Question', 'Option_1', 'Option_2', 'Option_3', 'Option_4']

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()

        # Get all PDFs in folder
        pdf_files = glob.glob(os.path.join(INPUT_FOLDER, "*.pdf"))

        if not pdf_files:
            print(f"No PDFs found in {INPUT_FOLDER}")
            return

        for pdf_path in pdf_files:
            filename = os.path.basename(pdf_path)
            print(f"\nProcessing: {filename}")

            # 1. OCR
            pages_data = get_text_from_pdf(pdf_path)

            full_text_combined = ""
            for _, text in pages_data:
                full_text_combined += text + "\n"

            # 2. Cleanup
            cleaned_lines = clean_ocr_text(full_text_combined)

            # 3. Parse
            questions = parse_questions(cleaned_lines, filename)

            # 4. Save to CSV
            if questions:
                writer.writerows(questions)
                print(f" -> Extracted {len(questions)} questions.")
            else:
                print(" -> No questions found (check OCR quality or markers).")

    print(f"\nDone! Data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


Processing: 1std Kan FL Part- 1 2025-26.pdf
 Converting 1std Kan FL Part- 1 2025-26.pdf to images...
 Extracting text from 104 pages...
 -> Extracted 88 questions.

Processing: 2nd  Kan FL 2025 Part-1 2025-26.pdf
 Converting 2nd  Kan FL 2025 Part-1 2025-26.pdf to images...
 Extracting text from 64 pages...
 -> Extracted 127 questions.

Processing: 3rd Kan FL  Part-1 2025 -26.pdf
 Converting 3rd Kan FL  Part-1 2025 -26.pdf to images...
 Extracting text from 72 pages...
 -> Extracted 204 questions.

Processing: 4th Kan FL Part-1  2025-26.pdf
 Converting 4th Kan FL Part-1  2025-26.pdf to images...
 Extracting text from 72 pages...
 -> Extracted 202 questions.

Processing: 5th Kan FL Part-1  2025 -26.pdf
 Converting 5th Kan FL Part-1  2025 -26.pdf to images...
 Extracting text from 72 pages...
 -> Extracted 226 questions.

Processing: 6th Kan FL Part-1 2025-26.pdf
 Converting 6th Kan FL Part-1 2025-26.pdf to images...
 Extracting text from 80 pages...
 -> Extracted 213 questions.

Process

In [ ]:
!pip install opencv-python numpy

In [ ]:
import os
import io
import re
import csv
import glob
import cv2
import numpy as np
from google.cloud import vision
from pdf2image import convert_from_path
from PIL import Image

# ================= CONFIGURATION =================
# 1. Path to your Google Cloud JSON key
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"

# 2. Folder containing your input PDFs
INPUT_FOLDER = "/content/drive/MyDrive/kannada-tb"

# 3. Output filename
OUTPUT_CSV = "/content/drive/MyDrive/newkannada_questions_cleaned.csv"

# ================= LOGIC CONSTANTS =================
START_MARKERS = [r'ಅಭ್ಯಾಸ', r'ಅಭ್ಯಾಸಗಳು']

# Removed 'ವ್ಯಾಕರಣ' so grammar is captured.
END_MARKERS = [
    r'^ಪಾಠ\s+[0-9೦-೯]+',
    r'^ಪದ್ಯ\s+[0-9೦-೯]+',
    r'^ಗದ್ಯ\s+[0-9೦-೯]+',
    r'^ಘಟಕ\s+[0-9೦-೯]+',
    r'^ಪೂರಕ\s+ಓದು',
    r'^ಯೋಜನೆ',
]

SECTION_HEADER_PATTERN = re.compile(r'^\s*([ಅ-ಋa-zA-Z]+)[\.\)]\s+(.*)')
QUESTION_PATTERN = re.compile(r'^\s*([\d0-9೦-೯]+)[\.\)]\s+(.*)')
OPTION_PATTERN = re.compile(r'[\(\[]([ಅ-ಋa-dA-D]+)[\)\]]\s+([^(\[]+)')

# Specific patterns to remove from text if they survive image cleaning
WATERMARK_TEXT_PATTERNS = [
    r'@KTBS',
    r'NOT TO BE REPUBLISHED',
    r'be republished',
    r'Govt of Karnataka',
    r'Government of Karnataka'
]

# ===============================================

def preprocess_image(pil_image):
    """
    Takes a PIL image, converts it to OpenCV format,
    and applies thresholding to remove light watermarks.
    """
    # 1. Convert PIL image to numpy array (OpenCV format)
    img_array = np.array(pil_image)

    # 2. Convert to Grayscale
    # (Handle RGB vs RGBA images)
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array # Already grayscale

    # 3. Apply Thresholding (Binarization)
    # Any pixel lighter than 180 becomes 255 (White).
    # Any pixel darker than 180 becomes 0 (Black).
    # Since watermarks are usually light grey (around 200-220), they turn White.
    # Text is usually dark (< 50), so it stays Black.
    _, binary = cv2.threshold(gray, 190, 255, cv2.THRESH_BINARY)

    # 4. Convert back to an image format that Google API accepts
    success, encoded_image = cv2.imencode('.jpg', binary)
    return encoded_image.tobytes()

def get_text_from_pdf(pdf_path):
    client = vision.ImageAnnotatorClient()
    text_data = []

    print(f" Converting {os.path.basename(pdf_path)} to images...")
    try:
        # DPI 300 is optimal for OCR
        images = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        print(f"Error converting PDF: {e}")
        return []

    print(f" Pre-processing images and OCR-ing {len(images)} pages...")
    for i, image in enumerate(images):

        # --- NEW: CLEAN IMAGE BEFORE SENDING TO GOOGLE ---
        cleaned_image_bytes = preprocess_image(image)
        # -------------------------------------------------

        image_obj = vision.Image(content=cleaned_image_bytes)

        response = client.document_text_detection(image=image_obj)
        if response.full_text_annotation:
            text_data.append((i + 1, response.full_text_annotation.text))

    return text_data

def clean_ocr_text(text):
    """ Removes OCR artifacts and specific watermark text patterns. """
    lines = text.split('\n')
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Remove empty or very short lines
        if len(line) < 2:
            continue

        # Remove explicit watermark text using Regex
        is_watermark = False
        for pattern in WATERMARK_TEXT_PATTERNS:
            if re.search(pattern, line, re.IGNORECASE):
                is_watermark = True
                break

        if not is_watermark:
            cleaned_lines.append(line)

    return cleaned_lines

def parse_questions(cleaned_lines, filename):
    extracted_data = []

    is_in_exercise = False
    current_context = "General"

    for line in cleaned_lines:
        # 1. Check for Exercise Start
        if any(re.search(m, line) for m in START_MARKERS):
            is_in_exercise = True
            current_context = "General Exercise"
            continue

        # 2. Check for Chapter End
        if any(re.search(m, line) for m in END_MARKERS):
            is_in_exercise = False
            current_context = "General"
            continue

        if not is_in_exercise:
            continue

        # 3. Detect Section Headers
        header_match = SECTION_HEADER_PATTERN.match(line)
        question_match = QUESTION_PATTERN.match(line)

        if header_match and not question_match:
            current_context = header_match.group(2).strip()
            continue

        # 4. Detect Specific Questions
        if question_match:
            q_num = question_match.group(1)
            q_text = question_match.group(2)

            # --- MCQ PARSING ---
            options = []
            opt_matches = OPTION_PATTERN.findall(q_text)

            clean_q_text = q_text
            if opt_matches:
                clean_q_text = re.split(r'[\(\[]([ಅ-ಋa-dA-D]+)[\)\]]', q_text)[0].strip()
                options = [f"({m[0]}) {m[1].strip()}" for m in opt_matches]

            # --- SINGLE WORD FIX ---
            final_question = clean_q_text
            if len(clean_q_text.split()) < 3 and current_context != "General":
                final_question = f"{current_context}: {clean_q_text}"

            row = {
                'Filename': filename,
                'Section_Context': current_context,
                'Q_No': q_num,
                'Question': final_question,
                'Option_1': options[0] if len(options) > 0 else "",
                'Option_2': options[1] if len(options) > 1 else "",
                'Option_3': options[2] if len(options) > 2 else "",
                'Option_4': options[3] if len(options) > 3 else "",
            }
            extracted_data.append(row)

    return extracted_data

def main():
    # Setup CSV file
    csv_headers = ['Filename', 'Section_Context', 'Q_No', 'Question', 'Option_1', 'Option_2', 'Option_3', 'Option_4']

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()

        pdf_files = glob.glob(os.path.join(INPUT_FOLDER, "*.pdf"))

        if not pdf_files:
            print(f"No PDFs found in {INPUT_FOLDER}")
            return

        for pdf_path in pdf_files:
            filename = os.path.basename(pdf_path)
            print(f"\nProcessing: {filename}")

            # 1. OCR (With Image Cleaning)
            pages_data = get_text_from_pdf(pdf_path)

            full_text_combined = ""
            for _, text in pages_data:
                full_text_combined += text + "\n"

            # 2. Cleanup (Text based)
            cleaned_lines = clean_ocr_text(full_text_combined)

            # 3. Parse
            questions = parse_questions(cleaned_lines, filename)

            # 4. Save to CSV
            if questions:
                writer.writerows(questions)
                print(f" -> Extracted {len(questions)} questions.")
            else:
                print(" -> No questions found.")

    print(f"\nDone! Data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


Processing: 1std Kan FL Part- 1 2025-26.pdf
 Converting 1std Kan FL Part- 1 2025-26.pdf to images...
 Pre-processing images and OCR-ing 104 pages...
 -> Extracted 87 questions.

Processing: 2nd  Kan FL 2025 Part-1 2025-26.pdf
 Converting 2nd  Kan FL 2025 Part-1 2025-26.pdf to images...
 Pre-processing images and OCR-ing 64 pages...
 -> Extracted 125 questions.

Processing: 3rd Kan FL  Part-1 2025 -26.pdf
 Converting 3rd Kan FL  Part-1 2025 -26.pdf to images...
 Pre-processing images and OCR-ing 72 pages...
 -> Extracted 209 questions.

Processing: 4th Kan FL Part-1  2025-26.pdf
 Converting 4th Kan FL Part-1  2025-26.pdf to images...
 Pre-processing images and OCR-ing 72 pages...
 -> Extracted 200 questions.

Processing: 5th Kan FL Part-1  2025 -26.pdf
 Converting 5th Kan FL Part-1  2025 -26.pdf to images...
 Pre-processing images and OCR-ing 72 pages...
 -> Extracted 197 questions.

Processing: 6th Kan FL Part-1 2025-26.pdf
 Converting 6th Kan FL Part-1 2025-26.pdf to images...
 Pre-p